# Arabic OCR Post-Correction — Colab Training

Finetunes **Qwen2.5-0.5B-Instruct** to repair Arabic OCR output.

Dataset, weights and checkpoints all live on Colab's disk, never yours. The
only artifact that leaves is the LoRA adapter, pushed to the HuggingFace Hub.

**Before you run:** Runtime → Change runtime type → **T4 GPU**.

Repo: https://github.com/Crypto47/arabic-ocr-post-correction

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# Colab ships torch already; add only what is missing.
!pip -q install "transformers>=4.44" "peft>=0.14" "trl>=0.20" \
                "datasets>=2.20" "accelerate>=0.33" pyyaml kagglehub
print("deps installed")

In [ ]:
import pathlib

PROJECT = "/content/arabic-ocr-post-correction"

# Idempotent: safe to re-run, and safe after a runtime restart (which resets
# the working directory to /content but leaves the clone on disk).
if not pathlib.Path(PROJECT).exists():
    !git clone -q https://github.com/Crypto47/arabic-ocr-post-correction.git {PROJECT}
%cd {PROJECT}
!ls

## 2. Kaggle credentials

`kagglehub` looks for credentials in this order: Colab secrets, the
`KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables, then
`~/.kaggle/kaggle.json`. If it finds none it will prompt you below.

Get a token at **kaggle.com → Settings → API → Create New Token**.

In [ ]:
import kagglehub

try:
    print("signed in as:", kagglehub.whoami()["username"])
except Exception:
    kagglehub.login()

## 3. Download the clean Arabic corpus

`dataset_download` fetches, unpacks, and returns the folder path — no manual
unzip step.

**This pipeline needs clean Arabic TEXT**, which it corrupts into
(noisy → clean) training pairs. Several Arabic datasets that sound textual are
actually scanned page *images*; those train an OCR engine, they cannot feed a
corrector. `azharhasannsaif/arabic-official-documents` is one of them — it is
tagged `data type > image`, so it will not work here.

Verified text corpora, with Kaggle's own usability rating:

| ref | size | usability | note |
|---|---|---|---|
| `mohamedbentalb/arasum` | 132 MB | 0.94 | **default** — news articles, clean, quick |
| `abedkhooli/arabic-bert-corpus` | 1.7 GB | 0.94 | scale up to this for the real run |
| `antcorpus/antcorpus` | 10 MB | 0.75 | tiny — fastest possible smoke test |
| `ahmedmohsen2002/tashkeela-clean...` | 176 MB | 0.82 | GPL-2 — avoid if weights may go commercial |

In [ ]:
DATASET = "mohamedbentalb/arasum"   # <-- swap here

DATA_DIR = kagglehub.dataset_download(DATASET)
print("downloaded and unpacked to:", DATA_DIR)

## 4. Inspect what actually arrived

Ten seconds here beats discovering the format halfway through a training run.

In [ ]:
import collections
from pathlib import Path

TEXT = {".txt", ".jsonl", ".json", ".csv", ".tsv", ".parquet"}
IMAGES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".gif", ".pdf"}

root = Path(DATA_DIR)
files = [p for p in root.rglob("*") if p.is_file()]
counts = collections.Counter(p.suffix.lower() or "(no extension)" for p in files)

print(f"{len(files)} files, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB\n")
for ext, n in counts.most_common(10):
    print(f"  {ext:<14} {n}")

if not (set(counts) & TEXT):
    print("\nNO TEXT FILES FOUND.")
    if set(counts) & IMAGES:
        print("This is an image dataset — it cannot feed this pipeline.")
    print("Go back to the cell above and pick a text corpus.")
else:
    biggest = max((p for p in files if p.suffix.lower() in TEXT),
                  key=lambda p: p.stat().st_size)
    print(f"\nlargest text file: {biggest.relative_to(root)} "
          f"({biggest.stat().st_size / 1e6:.1f} MB)")
    print("-" * 60)
    with biggest.open(encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            print(line[:300].rstrip())

## 5. Build (noisy → clean) training pairs

The builder reads `.txt` / `.jsonl` / `.json` / `.csv` / `.tsv` / `.parquet`
and auto-detects the text column, so the internal layout does not matter.
Corruption severity is sampled per example between `--min-rate` and
`--max-rate`, so one model sees clean scans and wrecked ones alike.

In [ ]:
!cd "{PROJECT}" && python src/build_dataset.py \
    --input "{DATA_DIR}" \
    --output data/processed \
    --max-pairs 60000 \
    --min-rate 0.04 --max-rate 0.18

In [ ]:
import json

with open(f"{PROJECT}/data/processed/train.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        print("NOISY:", r["messages"][0]["content"].split("\n\n")[-1])
        print("CLEAN:", r["messages"][1]["content"])
        print("rate :", r["noise_rate"], "\n")

## 6. Baseline — score the *untuned* model first

Run this before training. If you cannot say what the base model scored, you
cannot claim the finetune achieved anything.

In [ ]:
!cd "{PROJECT}" && python src/evaluate.py --limit 200 --out results/eval_base.json

## 7. Train

In [ ]:
!cd "{PROJECT}" && python src/train.py --config configs/qwen05b_lora.yaml

## 8. Score the finetuned model

In [ ]:
!cd "{PROJECT}" && python src/evaluate.py \
    --adapter outputs/arabic-ocr-post-correction/final \
    --limit 200 \
    --out results/eval_tuned.json

In [ ]:
import json

base = json.load(open(f"{PROJECT}/results/eval_base.json", encoding="utf-8"))
tuned = json.load(open(f"{PROJECT}/results/eval_tuned.json", encoding="utf-8"))

print(f"{'':<20}{'CER':>10}{'WER':>10}")
print("-" * 40)
print(f"{'raw OCR':<20}{base['baseline']['cer']:>10.4f}{base['baseline']['wer']:>10.4f}")
print(f"{'base 0.5B':<20}{base['corrected']['cer']:>10.4f}{base['corrected']['wer']:>10.4f}")
print(f"{'finetuned':<20}{tuned['corrected']['cer']:>10.4f}{tuned['corrected']['wer']:>10.4f}")
print("-" * 40)
print(f"CER reduction vs raw OCR: {tuned['cer_reduction_pct']:.1f}%")
print("\nPaste these into the README results table.")

## 9. Try it

In [ ]:
!cd "{PROJECT}" && python src/infer.py \
    --adapter outputs/arabic-ocr-post-correction/final \
    --text "العلم نور يضنء طزيق الإنسان في الحتياة، وآلجهل ظلام دام"

## 10. Publish the adapter to HuggingFace

The adapter is a few tens of MB. The base model is pulled from the Hub at load
time, so nothing large is ever stored or uploaded.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ID = "Crypto47/arabic-ocr-post-correction-0.5b"
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = f"{PROJECT}/outputs/arabic-ocr-post-correction/final"

model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE), ADAPTER)
model.push_to_hub(REPO_ID)
AutoTokenizer.from_pretrained(ADAPTER).push_to_hub(REPO_ID)
print(f"https://huggingface.co/{REPO_ID}")

---

**Last step, and skipping it wastes the run:** copy the numbers from section 8
into the README results table, and record which corpus you used in
`data/README.md`. The repo is what people read; the notebook is only how it
got made.